In [ ]:
import pandas as pd
import numpy as np
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load Data

In [ ]:
filename = 'OptionData.csv'
df = pd.read_csv(filename)
df.head()

### Split Data

In [ ]:
train_df = df[:-20000]
test_df = df[-20000:]

### Prepare the Training Data

In [ ]:
def prepareData(data_df):
    drop_list = ['CallValue', 'CallDelta', 
                'CallGamma', 'CallTheta', 'CallVega', 
                'CallRho', 'PutValue', 'PutDelta', 
                'PutGamma', 'PutTheta', 'PutVega', 'PutRho']

    input_data = data_df.copy(deep=True)
    input_data = input_data.drop(drop_list, axis=1)
    
    call_input_data = input_data.copy(deep=True)
    put_input_data = input_data.copy(deep=True)
    
    call_input_data['PutCall'] = 1
    put_input_data['PutCall'] = 0
    
    frames_list = [call_input_data, put_input_data]
    X = pd.concat(frames_list)
    
    label_call = data_df['CallValue'].tolist()
    label_put = data_df['PutValue'].tolist()
    
    y = label_call + label_put
    y = np.array(y)
    
    return X, y

In [ ]:
X, y = prepareData(train_df)

### Prepare the Test Data

In [ ]:
X_test, y_test = prepareData(test_df)

### Scale Data

In [ ]:
from sklearn.preprocessing import StandardScaler
standard_scalar = StandardScaler()
X[['Vol', 'Spot', 'T']] = standard_scalar.fit_transform(X[['Vol', 'Spot', 'T']])

In [ ]:
X_test[['Vol', 'Spot', 'T']] = standard_scalar.transform(X_test[['Vol', 'Spot', 'T']])

### Shuffle

In [ ]:
from sklearn.utils import shuffle

In [ ]:
X, y = shuffle(X, y, random_state=42)

### Build Model

In [ ]:
class Swish(nn.Module):
    def __init__(self):
        super(Swish, self).__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)

def serlu(x):
    return 1.10786 * F.relu(x) + 3.1326 * torch.min(x, torch.tensor(0.0)) * torch.exp(torch.min(x, torch.tensor(0.0)))

class SERLU(nn.Module):
    def __init__(self):
        super(SERLU, self).__init__()
        
    def forward(self, x):
        return serlu(x)
    
activation_functions = {
    'relu': nn.ReLU(),
    'sigmoid': nn.Sigmoid(),
    'tanh': nn.Tanh(),
    'leaky_relu': nn.LeakyReLU(),
    'elu': nn.ELU(),
    'prelu' : nn.PReLU(),
    'selu' : nn.SELU(),
    'gelu' : nn.GELU(),
    'swish' : Swish(),
    'serlu' : SERLU()
}

In [ ]:
no_epochs = 500
batch_size = 10000

In [ ]:
train_x = torch.Tensor(X.to_numpy()).to(device)
train_y = torch.Tensor(y).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=10000, 
                                           shuffle=True, drop_last=True)

test_x = torch.Tensor(X_test.to_numpy()).to(device)
test_y = torch.Tensor(y_test).to(device)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, 
                                          shuffle=False, drop_last=True)

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_errors.append(train_loss)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_errors.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    return history

In [ ]:
def evaluate_model(model, test_loader, loss_fn):
    model.eval()  # Set the model to evaluation mode
    test_loss = 0
    with torch.no_grad():  # No gradient computation
        for data, target in test_loader:
            output = model(data)
            y_pred = output.squeeze()
            test_loss += loss_fn(y_pred, target).item()  # Sum up batch loss

    test_loss /= len(test_loader.dataset)
    return test_loss

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, activation_name):
        super(NeuralNetwork, self).__init__()
        self.activation = activation_functions[activation_name]
        self.fc1 = nn.Linear(4, 50)
        self.fc2 = nn.Linear(50, 50)
        self.fc3 = nn.Linear(50, 1)

    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)
        return x

### Define Sensitivity MSE function

In [ ]:
def calculateGradMSE(standard_scalar,
                     dy_dx,
                     source_df):
    # Scaling
    scaled_grads = dy_dx / standard_scalar.scale_
   
    msedict = dict()
    sensdict = dict()
    
    # Delta
    call_delta_dnn = np.array(scaled_grads[:20000,1])
    call_delta = test_df['CallDelta'].to_numpy()
    
    sensdict['Call Delta DNN'] = call_delta_dnn
    sensdict['Call Delta'] = call_delta
    
    call_delta_mse = ((call_delta_dnn - call_delta)**2).mean()
    msedict['Call Delta MSE'] = call_delta_mse
    
    put_delta_dnn = np.array(scaled_grads[20000:,1])
    put_delta = test_df['PutDelta'].to_numpy()
    
    sensdict['Put Delta DNN'] = put_delta_dnn
    sensdict['Put Delta'] = put_delta
    
    put_delta_mse = ((put_delta_dnn - put_delta)**2).mean()
    msedict['Put Delta MSE'] = put_delta_mse
    
    # Vega
    # Note the raw vega for both BS QuantLib and DNN need to be scaled down by 100
    # to represent a 1% shift which is the normal quoting convention for Vega
    call_vega_dnn = np.array(scaled_grads[:20000,0]) / 100.0
    call_vega = test_df['CallVega'].to_numpy() / 100.0
    
    sensdict['Call Vega DNN'] = call_vega_dnn
    sensdict['Call Vega'] = call_vega
    
    call_vega_mse = ((call_vega_dnn - call_vega)**2).mean()
    msedict['Call Vega MSE'] = call_vega_mse
    
    put_vega_dnn = np.array(scaled_grads[20000:,0]) /100.0
    put_vega = test_df['PutVega'].to_numpy() / 100.0
    
    sensdict['Put Vega DNN'] = put_vega_dnn
    sensdict['Put Vega'] = put_vega
    
    put_vega_mse = ((put_vega_dnn - put_vega)**2).mean()
    msedict['Put Vega MSE'] = put_vega_mse
    
    # Theta
    # Note the raw theta in QuantLib and DNN is for 1 year - scale by 1.0 / 365.0
    # Note the raw theta in DNN has opposite sign convention to QuantLib
    call_theta_dnn = -np.array(scaled_grads[:20000,2]) / 365.0
    call_theta = test_df['CallTheta'].to_numpy() / 365.0
    
    sensdict['Call Theta DNN'] = call_theta_dnn
    sensdict['Call Theta'] = call_theta
    
    call_theta_mse = ((call_theta_dnn - call_theta)**2).mean()
    msedict['Call Theta MSE'] = call_theta_mse
    
    put_theta_dnn = -np.array(scaled_grads[20000:,2]) / 365.0
    put_theta = test_df['PutTheta'].to_numpy() / 365.0
    
    sensdict['Put Theta DNN'] = put_theta_dnn
    sensdict['Put Theta'] = put_theta
    
    put_theta_mse = ((put_theta_dnn - put_theta)**2).mean()
    msedict['Put Theta MSE'] = put_theta_mse
    
    return msedict, sensdict

### Evaluate Model

In [ ]:
def evaluateModel(activation, train_loader, test_loader, epochs):
    results_dict = dict()
    
    model = NeuralNetwork(activation).to(device)
    optimizer = optim.RMSprop(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    history = train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs)
    results_dict['history'] = history
    
    # Evaluate
    test_loss = evaluate_model(model, test_loader, loss_fn)
    results_dict['test_loss'] = test_loss
    
    X_input = torch.tensor(X_test.to_numpy(), dtype=torch.float32).to(device)
    # Enable gradient tracking for X_input
    X_input.requires_grad_(True)
    # Forward pass
    y_output = model(X_input)
    # Backward pass (calculate gradients)
    y_output.backward(torch.ones_like(y_output))
    # Extract gradients
    dy_dx = X_input.grad

    # Drop categorical variable
    dy_dx = dy_dx[:,:3].detach().cpu().numpy()
    
    msedict, sensdict = calculateGradMSE(standard_scalar, dy_dx, test_df)
    results_dict['mse_sens'] = msedict
    results_dict['sens_results'] = sensdict
    results_dict['Test_output'] = y_output.detach().cpu().numpy()
    results_dict['True_output'] = y_test
    return results_dict

### Plotting

In [ ]:
def plotValueDist(results_dict, activation):
    call_value_dnn = results_dict['Test_output'][:20000].reshape(20000)
    put_value_dnn = results_dict['Test_output'][20000:].reshape(20000)
    call_value = results_dict['True_output'][:20000].reshape(20000)
    put_value = results_dict['True_output'][20000:].reshape(20000)
    
    call_rel_err = (call_value_dnn - call_value) / 100.0
    put_rel_err = (put_value_dnn - put_value) / 100.0
    
    fig, axs = plt.subplots(1, 2, figsize=(10,5))

    axs[0].hist(call_rel_err, bins=100, range=(-0.03, 0.03), density=True, facecolor='k')
    axs[0].set_title("Call")
    axs[1].hist(put_rel_err, bins=100, range=(-0.03, 0.03), density=True, facecolor='k')
    axs[1].set_title("Put")

    for ax in axs.flat:
        ax.set(xlabel='Error', ylabel='Density')
        ax.grid(True)
        
    plt.savefig(activation + '_ValueDist.png', dpi=300, bbox_inches='tight')

In [ ]:
def plotSensDist(results_dict, activation):
    
    call_delta_dnn = results_dict['sens_results']['Call Delta DNN'].reshape(20000)
    put_delta_dnn = results_dict['sens_results']['Put Delta DNN'].reshape(20000)
    call_delta = results_dict['sens_results']['Call Delta'].reshape(20000)
    put_delta = results_dict['sens_results']['Put Delta'].reshape(20000)
    
    call_abs_delta_err = (call_delta_dnn - call_delta)
    put_abs_delta_err = (put_delta_dnn - put_delta)
    
    call_vega_dnn = results_dict['sens_results']['Call Vega DNN'].reshape(20000)
    put_vega_dnn = results_dict['sens_results']['Put Vega DNN'].reshape(20000)
    call_vega = results_dict['sens_results']['Call Vega'].reshape(20000)
    put_vega = results_dict['sens_results']['Put Vega'].reshape(20000)
    
    call_abs_vega_err = (call_vega_dnn - call_vega)
    put_abs_vega_err = (put_vega_dnn - put_vega)
    
    call_theta_dnn = results_dict['sens_results']['Call Theta DNN'].reshape(20000)
    put_theta_dnn = results_dict['sens_results']['Put Theta DNN'].reshape(20000)
    call_theta = results_dict['sens_results']['Call Theta'].reshape(20000)
    put_theta = results_dict['sens_results']['Put Theta'].reshape(20000)
    
    call_abs_theta_err = (call_theta_dnn - call_theta)
    put_abs_theta_err = (put_theta_dnn - put_theta)
    
    figrows = 3
    
    fig, axs = plt.subplots(figrows, 2, figsize=(10,figrows * 5))

    axs[0, 0].hist(call_abs_delta_err, bins=100, range=(-0.2, 0.2), density=True, facecolor='k')
    axs[0, 0].set_title("Call Delta")
    axs[0, 1].hist(put_abs_delta_err, bins=100, range=(-0.2, 0.2), density=True, facecolor='k')
    axs[0, 1].set_title("Put Delta")

    axs[1, 0].hist(call_abs_vega_err, bins=100, range=(-0.2, 0.2), density=True, facecolor='k')
    axs[1, 0].set_title("Call Vega")
    axs[1, 1].hist(put_abs_vega_err, bins=100, range=(-0.2, 0.2), density=True, facecolor='k')
    axs[1, 1].set_title("Put Vega")

    axs[2, 0].hist(call_abs_theta_err, bins=100, range=(-0.02, 0.02), density=True, facecolor='k')
    axs[2, 0].set_title("Call Theta")
    axs[2, 1].hist(put_abs_theta_err, bins=100, range=(-0.02, 0.02), density=True, facecolor='k')
    axs[2, 1].set_title("Put Theta")

    fig.tight_layout(pad=3.0)

    for ax in axs.flat:
        ax.set(xlabel='Error', ylabel='Density')
        ax.grid(True)
        
    plt.savefig(activation + '_SensDist.png', dpi=300, bbox_inches='tight')

### ReLU

In [ ]:
relu_results = evaluateModel('relu', train_loader, test_loader, no_epochs)

In [ ]:
print(relu_results['mse_sens'])

In [ ]:
plotValueDist(relu_results, 'relu')

In [ ]:
plotSensDist(relu_results, 'relu')

### Sigmoid

In [ ]:
sigmoid_results = evaluateModel('sigmoid', train_loader, test_loader, no_epochs)

In [ ]:
print(sigmoid_results['mse_sens'])

In [ ]:
plotValueDist(sigmoid_results, 'sigmoid')

In [ ]:
plotSensDist(sigmoid_results, 'sigmoid')

### Tanh

In [ ]:
tanh_results = evaluateModel('tanh', train_loader, test_loader, no_epochs)

In [ ]:
print(tanh_results['mse_sens'])

In [ ]:
plotValueDist(tanh_results, 'tanh')

In [ ]:
plotSensDist(tanh_results, 'tanh')

### ELU

In [ ]:
elu_results = evaluateModel('elu', train_loader, test_loader, no_epochs)

In [ ]:
print(elu_results['mse_sens'])

In [ ]:
plotValueDist(elu_results, 'elu')

In [ ]:
plotSensDist(elu_results, 'elu')

### Leaky ReLU

In [ ]:
leakyrelu_results = evaluateModel('leaky_relu', train_loader, test_loader, no_epochs)

In [ ]:
print(leakyrelu_results['mse_sens'])

In [ ]:
plotValueDist(leakyrelu_results, 'leakyrelu')

In [ ]:
plotSensDist(leakyrelu_results, 'leakyrelu')

### Swish

In [ ]:
swish_results = evaluateModel('swish', train_loader, test_loader, no_epochs)

In [ ]:
print(swish_results['mse_sens'])

In [ ]:
plotValueDist(swish_results, 'swish')

In [ ]:
plotSensDist(swish_results, 'swish')

### PReLU

In [ ]:
prelu_results = evaluateModel('prelu', train_loader, test_loader, no_epochs)

In [ ]:
print(prelu_results['mse_sens'])

In [ ]:
plotValueDist(prelu_results, 'prelu')

In [ ]:
plotSensDist(prelu_results, 'prelu')

### SELU

In [ ]:
selu_results = evaluateModel('selu', train_loader, test_loader, no_epochs)

In [ ]:
print(selu_results['mse_sens'])

In [ ]:
plotValueDist(selu_results, 'selu')

In [ ]:
plotSensDist(selu_results, 'selu')

### GELU

In [ ]:
gelu_results = evaluateModel('gelu', train_loader, test_loader, no_epochs)

In [ ]:
print(gelu_results['mse_sens'])

In [ ]:
plotValueDist(gelu_results, 'gelu')

In [ ]:
plotSensDist(gelu_results, 'gelu')

### SERLU

In [ ]:
serlu_results = evaluateModel('serlu', train_loader, test_loader, no_epochs)

In [ ]:
print(serlu_results['mse_sens'])

In [ ]:
plotValueDist(serlu_results, 'serlu')

In [ ]:
plotSensDist(serlu_results, 'serlu')

### Build Results Table

In [ ]:
results_dict = dict()

In [ ]:
results_dict['ReLU'] = relu_results['mse_sens']
results_dict['sigmoid'] = sigmoid_results['mse_sens']
results_dict['tanh'] = tanh_results['mse_sens']
results_dict['elu'] = elu_results['mse_sens']
results_dict['leakyrelu'] = leakyrelu_results['mse_sens']
results_dict['swish'] = swish_results['mse_sens']
results_dict['prelu'] = prelu_results['mse_sens']
results_dict['selu'] = selu_results['mse_sens']
results_dict['gelu'] = gelu_results['mse_sens']
results_dict['serlu'] = serlu_results['mse_sens']

results_dict['ReLU']['Loss'] = relu_results['test_loss']
results_dict['sigmoid']['Loss'] = sigmoid_results['test_loss']
results_dict['tanh']['Loss'] = tanh_results['test_loss']
results_dict['elu']['Loss'] = elu_results['test_loss']
results_dict['leakyrelu']['Loss'] = leakyrelu_results['test_loss']
results_dict['swish']['Loss'] = swish_results['test_loss']
results_dict['prelu']['Loss'] = prelu_results['test_loss']
results_dict['selu']['Loss'] = selu_results['test_loss']
results_dict['gelu']['Loss'] = gelu_results['test_loss']
results_dict['serlu']['Loss'] = serlu_results['test_loss']

In [ ]:
result_df = pd.DataFrame(results_dict)
result_df.head(10)

In [ ]:
result_df.to_latex()